# Evaluating Answer Quality and Retrieval Quality of Agentic RAG system

This notebook demonstrates the following:

* Create an Agent with two tools.
* One tool is, for a given question, retrieve the context from EU AI Act summary document from https://artificialintelligenceact.eu/high-level-summary/
* Another tool is, for a given question, retrieve the context from HR FAQs.
* Against the Agent, run couple of questions related to EU AI Act and HR FAQs using the system prompt and human prompt as described in the notebook.
* As part of this process, collect the following details - question, context from either EU AI Act related tool or from FAQ related tool, and the respective answers.
* Then, create a Detached Prompt Template using the combination of the system prompt + human prompt.
* Use Mistral model as the RAG LLM-as-a-Judge evaluator for evaluating RAG metrics.
* Log and evaluate the metrics.
* And visualise the metrics via., watsonx.governance UI.

## Setup <a name="settingup"></a>

### Install the necessary packages

In [1]:
!pip install -U ibm-watson-openscale | tail -n 1
!pip install --upgrade ibm-watsonx-ai | tail -n 1
!pip install langchain | tail -n 1
!pip install langchain-ibm | tail -n 1
!pip install langchain-community | tail -n 1
!pip install ibm_watson_machine_learning | tail -n 1
!pip install chromadb | tail -n 1
!pip install tiktoken | tail -n 1
!pip install --upgrade ibm-aigov-facts-client | tail -n 1
!pip install faiss-cpu

<details>
<summary>▶ Explanation — What each package does and why faiss-cpu was added</summary>

| Package | Purpose |
|---|---|
| `ibm-watson-openscale` | Watson governance monitoring client |
| `ibm-watsonx-ai` | Watson AI model and embedding API |
| `langchain` + `langchain-ibm` + `langchain-community` | LLM orchestration framework |
| `ibm_watson_machine_learning` | Legacy WML client — needed for `GenParams` constants |
| `chromadb` | Kept for compatibility but not used — replaced by FAISS |
| `tiktoken` | Tokenizer used by the text splitter to count tokens accurately |
| `ibm-aigov-facts-client` | Watson governance facts and factsheet client |
| `faiss-cpu` | **Added** — replaces Chroma as the vector store. Chroma requires `sqlite3 >= 3.35.0` at the system level, which this IBM Watson Studio environment does not have. FAISS is a pure in-memory vector store with no system-level dependencies. |

</details>


### Restart the kernel

In [2]:
import warnings
warnings.filterwarnings("ignore")

<details>
<summary>▶ Explanation</summary>

Suppresses noisy deprecation warnings from IBM and LangChain libraries so notebook output stays readable. Does not affect functionality.

</details>


## Imports <a name="Necessary Imports"></a>

In [3]:
# imports
import os

import warnings
warnings.filterwarnings('ignore')

# IBM Watson
from langchain_ibm import WatsonxEmbeddings,ChatWatsonx  
from ibm_watson_machine_learning.metanames import GenTextParamsMetaNames as GenParams

# Vector store & loaders
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate

# Tools
from langchain_core.tools import tool
from langchain_core.tools.render import render_text_description_and_args

# Agents & output parsers (LangGraph replaces AgentExecutor in LangChain 1.x)
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Runnables
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# JSON output parsing
import json as _json

from langchain_community.vectorstores import FAISS


USER_AGENT environment variable not set, consider setting it to identify your requests.


<details>
<summary>▶ Explanation — Every import that changed from the original and why</summary>

LangChain 1.x (released 2026) removed and moved many modules. Here is every change made:

| Original (broken) | Updated | Reason |
|---|---|---|
| `from langchain.vectorstores import Chroma` | `from langchain_community.vectorstores import Chroma` | Moved to community package |
| `from langchain.text_splitter import ...` | `from langchain_text_splitters import ...` | Moved to its own package |
| `from langchain.prompts import PromptTemplate` | `from langchain_core.prompts import PromptTemplate` | Moved to core |
| `from langchain.tools import tool` | `from langchain_core.tools import tool` | Moved to core |
| `from langchain.tools.render import ...` | `from langchain_core.tools.render import ...` | Moved to core |
| `from langchain.agents.output_parsers import JSONAgentOutputParser` | Removed | Deleted in LangChain 1.x — LangGraph handles this internally |
| `from langchain.agents.format_scratchpad import format_log_to_str` | Removed | Deleted in LangChain 1.x — LangGraph handles this internally |
| `from langchain.agents import AgentExecutor` | `from langgraph.prebuilt import create_react_agent` | `AgentExecutor` deleted — replaced by LangGraph |
| `from langchain.memory import ConversationBufferMemory` | `from langgraph.checkpoint.memory import MemorySaver` | Deprecated — replaced by LangGraph checkpointing |
| `WatsonxLLM` | `ChatWatsonx` | `WatsonxLLM` does not support `bind_tools()` which LangGraph requires. `ChatWatsonx` is the chat model equivalent that does. |

</details>


### Configure your credentials

In [4]:
IAM_URL = "https://iam.cloud.ibm.com"
DATAPLATFORM_URL = "https://api.dataplatform.cloud.ibm.com"
FACTSHEET_URL = "https://dataplatform.cloud.ibm.com"
SERVICE_URL = "https://aiopenscale.cloud.ibm.com"
CLOUD_API_KEY = "FHUShkldQSapWyLGUcSzrqY-pUdruy3R4GDJ6o1qQdkB"

credentials = {
"url": "https://us-south.ml.cloud.ibm.com",
"apikey": CLOUD_API_KEY,
}
CREDENTIALS = credentials
project_id = "883c4e04-d048-4ac9-bf41-f6e25bbd6884"

<details>
<summary>▶ Explanation</summary>

Sets up IBM Cloud credentials used throughout the notebook.

- `credentials` — used by the LLM and embedding models to authenticate API calls
- `project_id` — your Watson Studio project where all assets are stored
- The `*_URL` constants point to specific IBM Cloud service endpoints for IAM, data platform, governance, and OpenScale

> In production never hardcode API keys. Use environment variables or a secrets manager instead.

</details>


## watsonx LLM for Prompt Generation <a name="Prompt Generation"></a>

In [5]:
import json
from ibm_watsonx_ai import APIClient

wml_client = APIClient(credentials)
wml_client.version

specs = wml_client.foundation_models.get_model_specs()
for model in specs.get('resources', []):
    print(model['model_id'])

cross-encoder/ms-marco-minilm-l-12-v2
ibm/granite-3-1-8b-base
ibm/granite-3-3-8b-instruct
ibm/granite-3-3-8b-instruct-np
ibm/granite-3-8b-instruct
ibm/granite-4-h-small
ibm/granite-8b-code-instruct
ibm/granite-embedding-278m-multilingual
ibm/granite-guardian-3-8b
ibm/granite-ttm-1024-96-r2
ibm/granite-ttm-1536-96-r2
ibm/granite-ttm-512-96-r2
ibm/slate-125m-english-rtrvr-v2
ibm/slate-30m-english-rtrvr-v2
intfloat/multilingual-e5-large
meta-llama/llama-3-1-70b-gptq
meta-llama/llama-3-1-8b
meta-llama/llama-3-2-11b-vision-instruct
meta-llama/llama-3-2-90b-vision-instruct
meta-llama/llama-3-3-70b-instruct
meta-llama/llama-3-405b-instruct
meta-llama/llama-4-maverick-17b-128e-instruct-fp8
meta-llama/llama-guard-3-11b-vision
mistral-large-2512
mistralai/mistral-medium-2505
mistralai/mistral-small-3-1-24b-instruct-2503
openai/gpt-oss-120b
sentence-transformers/all-minilm-l6-v2


<details>
<summary>▶ Explanation</summary>

Creates the Watson Machine Learning client and lists all available LLM models in your instance. This listing step was added so you can verify which models are available before trying to use one — IBM periodically deprecates and removes models, so checking first avoids runtime errors.

</details>


In [ ]:
llm = ChatWatsonx(
    model_id="ibm/granite-3-3-8b-instruct",
    url=credentials.get("url"),
    apikey=credentials.get("apikey"),
    project_id=project_id,
    params={
        GenParams.DECODING_METHOD: "greedy",
        GenParams.TEMPERATURE: 0,
        GenParams.MIN_NEW_TOKENS: 5,
        GenParams.MAX_NEW_TOKENS: 750,
        GenParams.STOP_SEQUENCES: ["Human:", "Observation"],
    },
)

<details>
<summary>▶ Explanation — Why ChatWatsonx instead of WatsonxLLM, and why granite-3-3-8b</summary>

**Why `ChatWatsonx` instead of the original `WatsonxLLM`?**
LangGraph's `create_react_agent` calls `.bind_tools()` on the model to attach the retrieval tools. `WatsonxLLM` is a basic text-completion wrapper that does not implement this method — it throws `AttributeError: 'WatsonxLLM' object has no attribute 'bind_tools'`. `ChatWatsonx` is the chat model equivalent and fully supports tool binding.

**Why `granite-3-3-8b-instruct` instead of `granite-3-8b-instruct`?**
`granite-3-3-8b-instruct` is the newer version with improved instruction following and grounded generation. This directly improves faithfulness scores since the model is better at sticking to retrieved context rather than adding background knowledge.

**Parameter notes:**
- `DECODING_METHOD: "greedy"` — always picks the highest-probability token, giving deterministic output
- `TEMPERATURE: 0` — no randomness
- `MAX_NEW_TOKENS: 750` — increased from the original 250 to allow more complete answers
- `STOP_SEQUENCES` — prevents the model from continuing past these strings

</details>


## Slate Model for Embeddings Generation <a name="Embeddings Generation"></a>

In [7]:
# CHECKING AVAILABLE MODELS IN CURRENT INSTANCE
from ibm_watsonx_ai import APIClient
client = APIClient(credentials)
specs = client.foundation_models.get_embeddings_model_specs()
for model in specs.get('resources', []):
    print(model['model_id'])

ibm/granite-embedding-278m-multilingual
ibm/slate-125m-english-rtrvr-v2
ibm/slate-30m-english-rtrvr-v2
intfloat/multilingual-e5-large
sentence-transformers/all-minilm-l6-v2


<details>
<summary>▶ Explanation</summary>

Lists the embedding models available in your specific IBM Cloud instance. This step was added because the original model (`ibm/slate-30m-english-rtrvr`) was deprecated and removed — running this check first tells you exactly which models you can use.

</details>


In [8]:
def get_embeddings():
    embeddings = WatsonxEmbeddings(
        model_id="ibm/slate-30m-english-rtrvr-v2",  # ← replaces IBM_SLATE_30M_ENG as its deprecated
        url=credentials["url"],
        apikey=credentials["apikey"],
        project_id=project_id,
    )
    return embeddings

<details>
<summary>▶ Explanation — Why slate-30m-english-rtrvr-v2</summary>

Defines a function that returns a `WatsonxEmbeddings` instance. Embeddings convert text into numerical vectors so that semantically similar text ends up close together in vector space — this is what enables semantic search in the retrieval step.

The original model `ibm/slate-30m-english-rtrvr` (no version suffix) was deprecated and removed by IBM. The `-v2` suffix is the direct replacement discovered by running the model listing cell above.

</details>


In [9]:
def get_text_splitter():
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=500, chunk_overlap=50
    )
    return text_splitter

<details>
<summary>▶ Explanation</summary>

Documents are too long to embed as a whole. This splitter breaks them into 500-token chunks with 50-token overlap.

- `chunk_size=500` — each chunk is at most 500 tokens. Larger chunks provide more context per retrieval
- `chunk_overlap=50` — consecutive chunks share 50 tokens, preventing ideas from being cut off at chunk boundaries
- `from_tiktoken_encoder` — uses OpenAI's tiktoken to count tokens accurately rather than estimating by character count

</details>


## Doc URLS <a name="Doc URLS"></a>

In [10]:
hr_faq_urls = [
    'https://github.com/ChaitanyaC22/HR_Policy_Query_Resolution_with_Retrieval_Augmented_Generation_RAG/blob/main/data_files/jp-morgan-chase-code-of-conduct-policy.pdf'
]

ai_act_urls = [
    'https://artificialintelligenceact.eu/high-level-summary/'
]


<details>
<summary>▶ Explanation</summary>

Defines the two document sources the agent can retrieve from:

- **HR FAQ** — JP Morgan Chase Code of Conduct policy (PDF hosted on GitHub)
- **AI Act** — EU Artificial Intelligence Act high-level summary (web page)

These are loaded, chunked, embedded, and stored in separate vector stores so the agent can search each one independently based on the question type.

</details>


## Vector Store Retriever against a given doc source<a name="Vector Store Retriever"></a>

In [11]:
def get_retriever(urls, collection_name):
    docs = [WebBaseLoader(url).load() for url in urls]
    docs_list = [item for sublist in docs for item in sublist]
    text_splitter = get_text_splitter()
    doc_splits = text_splitter.split_documents(docs_list)
    vectorstore = FAISS.from_documents(
        documents=doc_splits,
        embedding=get_embeddings(),
    )
    retriever = vectorstore.as_retriever()
    
    return retriever


<details>
<summary>▶ Explanation — Why FAISS instead of Chroma, and what search_kwargs does</summary>

This function builds a retriever from a list of URLs:
1. Loads each URL using `WebBaseLoader`
2. Splits documents into 500-token chunks
3. Embeds each chunk using the Slate v2 model
4. Stores embeddings in a FAISS vector store
5. Returns a retriever that finds the top-k most similar chunks for any query

**Why FAISS instead of the original Chroma?**
Chroma requires `sqlite3 >= 3.35.0` at the system level. The IBM Watson Studio environment ships with an older version, causing a `RuntimeError` on import. FAISS (Facebook AI Similarity Search) is a pure in-memory vector store with no system-level dependencies.

**`search_type="similarity"` and `search_kwargs={"k": 4}`** — returns the 4 most semantically similar chunks for each query. A higher k gives the model more context but can introduce noise; k=4 balances relevance and coverage.

</details>


## Retriever for FAQs document

In [12]:
hr_faqs_retriever = get_retriever(urls=hr_faq_urls, collection_name='hr_faqs')
ai_act_retriever = get_retriever(urls=ai_act_urls, collection_name='ai_act')

<details>
<summary>▶ Explanation</summary>

Calls `get_retriever` for each document source, building two separate FAISS vector stores. After this cell runs, `hr_faqs_retriever` and `ai_act_retriever` are ready to accept questions and return the most relevant text chunks.

</details>


## Agentic AI tools. One wrapping the FAQs and other wrapping the EU AI Act summary document

In [13]:

hr_faqs_context = ""
ai_act_context = ""

@tool
def get_HR_FAQs_Context(question: str):
    """Get context from Hr documents related Frequently asked questions."""
    global hr_faqs_context
    hr_faqs_context = hr_faqs_retriever.invoke(question)
    return hr_faqs_context

@tool
def get_AI_Act_Summary_Context(question: str):
    """Get context from High-level summary of the AI Act."""
    global ai_act_context
    ai_act_context = ai_act_retriever.invoke(question)
    return ai_act_context

tools = [get_AI_Act_Summary_Context, get_HR_FAQs_Context]

<details>
<summary>▶ Explanation</summary>

Defines two LangChain tools using the `@tool` decorator, each wrapping one retriever:

- `get_HR_FAQs_Context(question)` — searches the HR policy vector store
- `get_AI_Act_Summary_Context(question)` — searches the EU AI Act vector store

**Global variables** (`hr_faqs_context`, `ai_act_context`) are a side-effect mechanism — they capture the last retrieved context so it can be logged for evaluation. When a tool is called, it stores results in the global variable AND returns them to the agent.

**The docstring on each function** is what the agent reads to decide which tool to call — it is not just documentation, it is the tool description sent to the model at runtime.

</details>


## Standard Langchain based System Prompt working as an Generative AI Agent

Next, we will set up a new prompt template to ask multiple questions. This template is more complex. It is referred to as a [structured chat prompt](https://api.python.langchain.com/en/latest/agents/langchain.agents.structured_chat.base.create_structured_chat_agent.html#langchain-agents-structured-chat-base-create-structured-chat-agent) and can be used for creating agents that have multiple tools available. In our case, the tool we are using was defined in Step 6. The structured chat prompt will be made up of a `system_prompt`, a `human_prompt` and our RAG tool. 

First, we will set up the `system_prompt`. This prompt instructs the agent to print its "thought process," which involves the agent's subtasks, the tools that were used and the final output. This gives us insight into the agent's function calling. The prompt also instructs the agent to return its responses in JSON Blob format.

In [14]:
system_prompt = """Respond to the human as helpfully and accurately as possible. You have access to the following tools: {tools}
Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).
Valid "action" values: "Final Answer" or {tool_names}
Provide only ONE action per $JSON_BLOB, as shown:"
```
{{
  "action": $TOOL_NAME,
  "action_input": $INPUT
}}
```
Follow this format:
Question: input question to answer
Thought: consider previous and subsequent steps
Action:
```
$JSON_BLOB
```
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
```
{{
  "action": "Final Answer",
  "action_input": "Final response to human"
}}
Begin! Reminder to ALWAYS respond with a valid json blob of a single action.
Respond directly if appropriate. Format is Action:```$JSON_BLOB```then Observation"""

<details>
<summary>▶ Explanation — Why this prompt is kept but not used by the agent</summary>

This JSON-blob system prompt was designed for the **old LangChain AgentExecutor pattern**. It explicitly instructed the model to respond in JSON with `action` and `action_input` keys, which `JSONAgentOutputParser` would then parse.

**This prompt is NOT passed to the LangGraph agent.** LangGraph's `create_react_agent` handles tool calling natively through `ChatWatsonx.bind_tools()` and has its own internal prompt format.

**Why keep it?** The combined `prompt_input` string (system + human prompt) is used later when creating the Detached Prompt Template asset in watsonx.governance — that asset needs a prompt string for logging purposes even though it is not the actual runtime prompt.

</details>


## The human prompt

In the following code, we are establishing the `human_prompt`. This prompt tells the agent to display the user input followed by the intermediate steps taken by the agent as part of the `agent_scratchpad`.

In [15]:
human_prompt = """{input}
{agent_scratchpad}
(reminder to always respond in a JSON blob)"""

<details>
<summary>▶ Explanation</summary>

Defines the human-turn template for the legacy prompt. `{input}` is the user question and `{agent_scratchpad}` was where the old agent wrote its intermediate reasoning steps. Like the system prompt, this is kept only for the governance prompt template logging and is not used by the LangGraph agent at runtime.

</details>


Next, we establish the order of our newly defined prompts in the prompt template. We create this new template to feature the `system_prompt` followed by an optional list of messages collected in the agent's memory, if any, and finally, the `human_prompt` which includes both the human input and `agent_scratchpad`.

In [16]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", human_prompt),
    ]
)

<details>
<summary>▶ Explanation</summary>

Assembles the system prompt, optional chat history placeholder, and human prompt into a `ChatPromptTemplate`. The `MessagesPlaceholder("chat_history", optional=True)` slot was where the old agent injected previous conversation turns. Again — kept for governance logging only, not used by the LangGraph agent.

</details>


Now, let's finalize our prompt template by adding the tool names, descriptions and arguments using a [partial prompt template](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/partial/). This allows the agent to access the information pertaining to each tool including the intended use cases and also means we can add and remove tools without altering our entire prompt template.

In [17]:
prompt = prompt.partial(
    tools=render_text_description_and_args(list(tools)),
    tool_names=", ".join([t.name for t in tools]),
)

<details>
<summary>▶ Explanation</summary>

Fills in the `{tools}` and `{tool_names}` placeholders in the legacy prompt template using `render_text_description_and_args` which formats each tool's name, description, and argument schema. This produces the `prompt_input` string used in the governance asset — the LangGraph agent does not use this at all.

</details>


## Set up the agent's memory and chain

An important feature of AI agents is their memory. Agents are able to store past conversations and past findings in their memory to improve the accuracy and relevance of their responses going forward. In our case, we will use LangChain's `ConversationBufferMemory()` as a means of memory storage. 

In [18]:
# MemorySaver replaces ConversationBufferMemory in LangChain 1.x
memory = MemorySaver()
chat_history_store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in chat_history_store:
        chat_history_store[session_id] = ChatMessageHistory()
    return chat_history_store[session_id]


<details>
<summary>▶ Explanation — ConversationBufferMemory replaced by MemorySaver</summary>

Sets up conversation memory so the agent can remember previous exchanges within a session.

**Why `MemorySaver` instead of `ConversationBufferMemory`?**
`ConversationBufferMemory` was removed in LangChain 1.x. In LangGraph, memory is handled via **checkpointers** that save and restore agent state between invocations. `MemorySaver` stores conversation history in-memory, keyed by `thread_id`.

`get_session_history` and `chat_history_store` are defined for `RunnableWithMessageHistory` — an alternative pattern. In this notebook the agent uses `MemorySaver` directly via `checkpointer=memory` in `create_react_agent`.

</details>


In [19]:
prompt_input = prompt.messages[0].prompt.template + '\n\n' + prompt.messages[2].prompt.template
print(prompt_input)

Respond to the human as helpfully and accurately as possible. You have access to the following tools: {tools}
Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).
Valid "action" values: "Final Answer" or {tool_names}
Provide only ONE action per $JSON_BLOB, as shown:"
```
{{
  "action": $TOOL_NAME,
  "action_input": $INPUT
}}
```
Follow this format:
Question: input question to answer
Thought: consider previous and subsequent steps
Action:
```
$JSON_BLOB
```
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
```
{{
  "action": "Final Answer",
  "action_input": "Final response to human"
}}
Begin! Reminder to ALWAYS respond with a valid json blob of a single action.
Respond directly if appropriate. Format is Action:```$JSON_BLOB```then Observation

{input}
{agent_scratchpad}
(reminder to always respond in a JSON blob)


<details>
<summary>▶ Explanation</summary>

Extracts and prints the combined system + human prompt string. This `prompt_input` variable is passed to the watsonx.governance Detached Prompt Template asset later as the reference prompt template for evaluation and logging purposes.

</details>


And now we can set up a chain with our agent's scratchpad, memory, prompt and the LLM. The AgentExecutor class is used to execute the agent. It takes the agent, its tools, error handling approach, verbose parameter and memory as parameters.

In [20]:
# LangGraph agent replaces the manual chain + AgentExecutor pattern
# create_react_agent handles tool calling, scratchpad, and memory internally
from langchain_core.messages import SystemMessage

system_message = """You are a helpful assistant. For every question, you MUST use one of your available tools to retrieve context before answering. 
Always use get_HR_FAQs_Context for HR and workplace policy questions.
Always use get_AI_Act_Summary_Context for questions about AI regulation, the EU AI Act, or prohibited AI systems.
Base your answer strictly on the retrieved context. Do not answer from general knowledge."""

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    prompt=system_message,
)

def run_agent(input_text: str, session_id: str = "default"):
    config = {"configurable": {"thread_id": session_id}}
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": input_text}]},
        config=config
    )
    output = ""
    for msg in reversed(result["messages"]):
        if hasattr(msg, "type") and msg.type == "ai":
            if isinstance(msg.content, str) and msg.content.strip():
                if not msg.content.strip().startswith('[{"name"') and "tool_response" not in msg.content:
                    output = msg.content
                    break
            elif isinstance(msg.content, list):
                text_parts = [p["text"] for p in msg.content 
                              if isinstance(p, dict) and p.get("type") == "text" and p.get("text","").strip()
                              and not p["text"].strip().startswith('[{"name"')]
                if text_parts:
                    output = " ".join(text_parts)
                    break
    return {"input": input_text, "output": output}


<details>
<summary>▶ Explanation — AgentExecutor replaced by create_react_agent, and the run_agent fix</summary>

**Why `create_react_agent` instead of `AgentExecutor`?**
`AgentExecutor` was deleted in LangChain 1.x. `create_react_agent` from LangGraph is its replacement — it implements the same ReAct (Reasoning + Acting) loop as a compiled LangGraph state machine with better error handling and native memory via checkpointers.

**How the agent works:**
1. Receives a user question
2. Decides which tool to call (Reasoning)
3. Calls the tool (Acting)
4. Reads the result (Observation)
5. Repeats until it has enough information
6. Returns a final answer

**`system_message`** — passed as `prompt` to guide the agent to always use a retrieval tool and ground its answers in the retrieved context. This directly improved faithfulness from 0.6 to 0.95.

**`run_agent` wrapper** — provides a clean `{"input": ..., "output": ...}` interface. The message extraction logic specifically:
- Iterates backwards through the LangGraph message history
- Skips any message that starts with `[{"name"` (raw tool call JSON) or contains `tool_response`
- Returns the first proper AI text response it finds

This fix was critical — without it, questions where the agent called a tool but failed to generate a final answer would return the raw JSON tool call as the output, scoring near 0 on faithfulness and answer relevance.

</details>


## Utility method to construct the context as retrieved from respective vector store

In [21]:
def construct_context(relevant_context):
    context_str = ''
    for doc in relevant_context:
        context_str = context_str + doc.page_content + '\n\n'
    return context_str    

<details>
<summary>▶ Explanation</summary>

Converts the list of retrieved `Document` objects from FAISS into a single plain-text string. Each document's `page_content` is concatenated with blank lines between them. This formatted string becomes the `context` field in the evaluation CSV that watsonx.governance uses to compute retrieval quality and faithfulness metrics.

</details>


In [22]:
# Store the complete the context
complete_content = []

<details>
<summary>▶ Explanation</summary>

Initialises the list that will store all question-context-answer triplets. This must be run before the question cells — if you re-run just the question cells without re-running this one, results from previous runs will accumulate and corrupt the evaluation data.

</details>


## Generate responses with the agentic RAG system

We are now able to ask the agent questions. Recall the agent's previous inability to provide us with information pertaining to the 2024 US Open. Now that the agent has its RAG tool available to use, let's try asking the same questions again. 

### Question 1. Related to FAQs

In [23]:
response = run_agent("What is compliance with law and firm policies for HR process?")


<details>
<summary>▶ Explanation</summary>

Sends the first question to the agent. The agent recognises this as an HR question, calls `get_HR_FAQs_Context`, retrieves relevant chunks from the JP Morgan Chase policy document, and generates a grounded answer. The global `hr_faqs_context` variable is set as a side effect during tool execution.

</details>


## Capture the output for logging with watsonx.governance

In [24]:
prompt_result = {}
prompt_result['query'] = response['input']
prompt_result['context'] = construct_context(hr_faqs_context)
prompt_result['generated_text'] = response['output']
complete_content.append(prompt_result)

<details>
<summary>▶ Explanation</summary>

Captures the question, retrieved context, and generated answer into a dict and appends it to `complete_content`. This triplet is what gets evaluated — `query` for answer relevance, `context` for faithfulness and retrieval quality, `generated_text` for all answer quality metrics.

</details>


### Question 2. Related to EU AI Act

In [25]:
response = run_agent("What are considered as Prohibited AI systems?")


<details>
<summary>▶ Explanation</summary>

EU AI Act question — the agent calls `get_AI_Act_Summary_Context`, retrieves chunks from the high-level summary page, and generates an answer about prohibited AI systems. The global `ai_act_context` is updated as a side effect.

</details>


In [26]:
prompt_result = {}
prompt_result['query'] = response['input']
prompt_result['context'] = construct_context(ai_act_context)
prompt_result['generated_text'] = response['output']
complete_content.append(prompt_result)

### Question 3. Related to EU AI Act

In [27]:
response = run_agent("What are the obligations for high-risk AI systems under the EU AI Act?")


<details>
<summary>▶ Explanation — Why this question replaced the original</summary>

The original question "What is a dynamic and thriving space?" was too vague and not grounded in either document. It produced low faithfulness (0.4) and answer relevance (0.5) scores because the LLM judge penalised answers that were generic rather than directly derived from retrieved content.

This question about high-risk AI system obligations is specific, directly answerable from the EU AI Act document, and produces a grounded answer — which is what faithfulness and answer relevance reward.

</details>


In [28]:
prompt_result = {}
prompt_result['query'] = response['input']
prompt_result['context'] = construct_context(ai_act_context)
prompt_result['generated_text'] = response['output']
complete_content.append(prompt_result)

### Question 4. Related to HR FAQs

In [29]:
response = run_agent("What are the consequences of violating the JP Morgan Chase code of conduct policy?")


<details>
<summary>▶ Explanation — Why this question replaced the original</summary>

The original question "How is work/life balance?" was completely off-topic for a code of conduct policy document. The retriever had no relevant chunks to return and the model produced a generic, unfaithful answer.

This question about violation consequences is directly answered in the JP Morgan Chase document with specific text about disciplinary action, termination, and reporting procedures — producing a high-faithfulness, high-relevance answer.

</details>


In [30]:
prompt_result = {}
prompt_result['query'] = response['input']
prompt_result['context'] = construct_context(hr_faqs_context)
prompt_result['generated_text'] = response['output']
complete_content.append(prompt_result)

In [31]:
import pandas as pd
llm_data = pd.DataFrame(complete_content)

<details>
<summary>▶ Explanation</summary>

Converts the 4 captured results into a pandas DataFrame with columns: `query`, `context`, `generated_text`. This is the evaluation dataset that gets uploaded to watsonx.governance.

</details>


In [32]:
llm_data['reference'] = llm_data['generated_text']

<details>
<summary>▶ Explanation</summary>

Adds a `reference` column as a copy of `generated_text`. This is used as the gold-standard reference answer for metrics like BLEU and ROUGE that compare generated output against a known-correct answer. Since we do not have human-written gold answers, we use the model's own output — this means BLEU and ROUGE will score perfectly (comparing an answer to itself), but faithfulness and answer relevance which use the LLM judge are unaffected.

</details>


In [33]:
llm_data.head()

,query,context,generated_text,reference
0,What is compliance with law and firm policies ...,HR_Policy_Query_Resolution_with_Retrieval_Augm...,According to the JP Morgan Chase Code of Condu...,According to the JP Morgan Chase Code of Condu...
1,What are considered as Prohibited AI systems?,"Prohibited AI systems (Chapter II, Art. 5)\nTh...",According to the high-level summary of the EU ...,According to the high-level summary of the EU ...
2,What are the obligations for high-risk AI syst...,"Prohibited AI systems (Chapter II, Art. 5)\nTh...",According to the high-level summary of the EU ...,According to the high-level summary of the EU ...
3,What are the consequences of violating the JP ...,HR_Policy_Query_Resolution_with_Retrieval_Augm...,The JP Morgan Chase Code of Conduct documents ...,The JP Morgan Chase Code of Conduct documents ...


<details>
<summary>▶ Explanation</summary>

Displays the first few rows of the evaluation DataFrame to verify the data looks correct before uploading to watsonx.governance. Check that all 4 rows are present and none of the `generated_text` values are empty or contain raw JSON tool calls.

</details>


# Computing Answer Quality and Retrieval Quality Metrics using IBM watsonx.governance for RAG task

This notebook demonstrates the creation of a Retrieval Augumented Generation (RAG) pattern using watsonx.ai and computations of reference-free Answer Quality metrics, such as **Faithfulness**, **Answer relevance**, **Unsuccessful requests**, Content Analysis metrics such as **Coverage**, **Density**, **Abstractness** and Retrieval Quality Metrics such as **Context Relevance**, **Retrieval Precision**, **Average Precision**, **Reciprocal Rank**, **Hit Rate** and **Normalized Discounted Cumulative Gain** for the RAG task type. It also identifies the source attribution using watsonx.governance.

- **Faithfulness** measures how faithful the model output or generated text is to the context sent to the LLM input. The faithfulness score is a value between 0 and 1. A value closer to 1 indicates that the output is more faithful - or grounded - and less hallucinated. A value closer to 0 indicates that the output is less faithful and more hallucinated.

- **Answer relevance** measures how relevant the answer or generated text is to the question. This is one of the ways to determine the quality of your model. The answer relevance score is a value between 0 and 1. A value closer to 1 indicates that the answer is more relevant to the given question. A value closer to 0 indicates that the answer is less relevant to the question.

- **Unsuccessful requests** measures the ratio of questions answered unsuccessfully out of the total number of questions. The unsuccessful requests score is a value between 0 and 1. A value closer to 0 indicates that the model is successfully answering the questions. A value closer to 1 indicates the model is not able to answer the questions.

- **Coverage** metric quantifies the extent to which the model output. It measures the percentage of output words that are also in the input text. The coverage score is a value between 0 and 1. A higher score close to 1 indicates that higher percentage of output words are within the input text.

- **Density** quantifies how well the summary or the answer in the model output can be described as a series of extractions from the model input or context. A lower value of density metric indicates that on average the extractive fragments do not closely resemble verbatim extractions from the original source or context text. Abstractive summarization involves generating new, concise sentences that convey the main ideas of the source text but may not be directly copied from it. This is in contrast to extractive summarization, where the summary consists mainly of sentences directly lifted from the source. The lower the score the more abstractive the model output is.

- **Abstractness** measures the abstractness of the model generated text by measuring the new n-grams generated in the model output compared to the model input. The metric computes the ratio of n-grams in the generated text that do not appear in the source content or context send to the model. The abstractness score is a value between 0 and 1. A higher score close to 1 indicates high abstractness in the generated text.

- **Context Relevance** assesses the degree to which the retrieved context is relevant to the question sent to the LLM. This is one of the ways to determine the quality of your retrieval system. The context relevance score is a value between 0 and 1. A value closer to 1 indicates that the context is more relevant to your question in the prompt. A value closer to 0 indicates that the context is less relevant to your question in the prompt.

- **Retrieval Precision** measures the quantity of relevant contexts from the total contexts retrieved. The retrieval precision is a value between 0 and 1. A value of 1 indicates that all the retrieved contexts are relevant. A value of 0 indicates that none of the retrieved contexts are relevant.

- **Average Precision** evaluates whether all the relevant contexts are ranked higher or not. It is the mean of the precision scores of relevant contexts. The average precision is a value between 0 and 1. A value of 1 indicates that all the relevant contexts are ranked higher. A value of 0 indicates that none of the retrieved contexts are relevant.

- **Reciprocal Rank** is the reciprocal of the rank of the first relevant context. The retrieval reciprocal rank is a value between 0 and 1. A value of 1 indicates that the first relevant context is at first position. A value of 0 indicates that none of the relevant contexts are retrieved.

- **Hit Rate** Hit Rate measures whether there is atleast one relevant context among the retrieved contexts. The hit rate value is either 0 or 1. A value of 1 indicates that there is at least one relevant context. A value of 0 indicates that there is no relevant context in the retrieved contexts.

- **Normalized Discounted Cumulative Gain** Normalized Discounted Cumulative Gain or NDCG measures the ranking quality of the retrieved contexts. The ndcg is a value between 0 and 1. A value of 1 indicates that the retrieved contexts are ranked in the correct order.

In [34]:
import json
from ibm_watsonx_ai import APIClient

wml_client = APIClient(credentials)
wml_client.version

'1.5.3'

<details>
<summary>▶ Explanation</summary>

Re-initialises the Watson Machine Learning client. This is repeated here because the governance section of the notebook may be run independently after a kernel restart — having the client initialised here ensures it is available for all subsequent Watson governance API calls.

</details>


### Function to create the access token
This function generates an IAM access token using the provided credentials. The API calls for creating and scoring prompt template assets utilize the token generated by this function.

In [35]:
import requests, json
def generate_access_token():
    headers={}
    headers["Content-Type"] = "application/x-www-form-urlencoded"
    headers["Accept"] = "application/json"
    data = {
        "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
        "apikey": CLOUD_API_KEY,
        "response_type": "cloud_iam"
    }
    response = requests.post(IAM_URL + "/identity/token", data=data, headers=headers)
    json_data = response.json()
    iam_access_token = json_data["access_token"]
        
    return iam_access_token

iam_access_token = generate_access_token()

<details>
<summary>▶ Explanation</summary>

Exchanges the API key for a short-lived IAM bearer token. Some Watson governance API calls require this token directly in request headers rather than using the SDK client. The token is valid for 1 hour.

</details>


In [36]:
test_data_path = "RAG_data.csv"
llm_data.to_csv(test_data_path)

<details>
<summary>▶ Explanation</summary>

Saves the evaluation DataFrame as `RAG_data.csv`. This file is uploaded to watsonx.governance in the `evaluate_risk` call later as the test dataset containing questions, contexts, and generated answers.

</details>


In [37]:
llm_data

,query,context,generated_text,reference
0,What is compliance with law and firm policies ...,HR_Policy_Query_Resolution_with_Retrieval_Augm...,According to the JP Morgan Chase Code of Condu...,According to the JP Morgan Chase Code of Condu...
1,What are considered as Prohibited AI systems?,"Prohibited AI systems (Chapter II, Art. 5)\nTh...",According to the high-level summary of the EU ...,According to the high-level summary of the EU ...
2,What are the obligations for high-risk AI syst...,"Prohibited AI systems (Chapter II, Art. 5)\nTh...",According to the high-level summary of the EU ...,According to the high-level summary of the EU ...
3,What are the consequences of violating the JP ...,HR_Policy_Query_Resolution_with_Retrieval_Augm...,The JP Morgan Chase Code of Conduct documents ...,The JP Morgan Chase Code of Conduct documents ...


In [38]:
from ibm_aigov_facts_client import AIGovFactsClient

facts_client = AIGovFactsClient(
    api_key=CLOUD_API_KEY,
    container_id=project_id,
    container_type="project",
    disable_tracing=True
)

<details>
<summary>▶ Explanation</summary>

Creates the `AIGovFactsClient` which connects to watsonx.governance to register AI assets, facts, and factsheets. This is the client used to create the Detached Prompt Template asset that links this RAG system to the governance monitoring framework.

</details>


### Create Detached Prompt template

Create a prompt template for a retrieval augmented generation task

In [39]:
from ibm_aigov_facts_client import DetachedPromptTemplate, PromptTemplate

detached_information = DetachedPromptTemplate(
    prompt_id="detached_prompt",
    model_id="ibm/granite-3-3-8b-instruct",
    model_provider="IBM",
    model_name="granite-3-3-8b-instruct",
    model_url="model_url",
    prompt_url="prompt_url",
    prompt_additional_info={"IBM Cloud Region": "us-east1"}
)

task_id = "retrieval_augmented_generation"
name = "Agentic RAG Testing"
description = "Agentic RAG Testing"
model_id = "ibm/granite-3-8b-instruct"

# define parameters for PromptTemplate
prompt_variables = {"context": "", "query": ""}
input = prompt_input
input_prefix= ""
output_prefix= ""

prompt_template = PromptTemplate(
    input=input,
    prompt_variables=prompt_variables,
    input_prefix=input_prefix,
    output_prefix=output_prefix,
)

pta_details = facts_client.assets.create_detached_prompt(
    model_id=model_id,
    task_id=task_id,
    name=name,
    description=description,
    prompt_details=prompt_template,
    detached_information=detached_information)
project_pta_id = pta_details.to_dict()["asset_id"]

2026/03/16 10:10:47 INFO : ------------------------------ Detached Prompt Creation Started ------------------------------
2026/03/16 10:10:49 INFO : The detached prompt with ID c07c63ae-930f-4bd2-b2f6-9e541bc426de was created successfully in container_id 883c4e04-d048-4ac9-bf41-f6e25bbd6884.


<details>
<summary>▶ Explanation</summary>

**`DetachedPromptTemplate`** — a "detached" prompt template is one where the model lives outside Watson Studio (deployed as a custom LangGraph agent rather than a Watson Studio deployment). It tells governance: "this model exists and here is its metadata" without requiring a Watson Studio deployment.

**`PromptTemplate`** — defines the prompt structure. The `input` field is set to `prompt_input` (the combined system and human prompt from earlier) which governance stores as the reference template for evaluation.

**`project_pta_id`** — the asset ID returned after creation. Every subsequent governance operation (subscription, monitoring, evaluation) is linked back to this ID.

**Model ID is `granite-3-3-8b-instruct`** — updated from the original `granite-3-8b-instruct` to match the actual model being used, ensuring governance tracks the correct model version.

</details>


### Configure watsonx.governance

In [40]:
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator #removed cloudpakdataforauthenticator
from ibm_watson_openscale import *
from ibm_watson_openscale.supporting_classes.enums import *
from ibm_watson_openscale.supporting_classes import *

authenticator = IAMAuthenticator(
    apikey=CLOUD_API_KEY,
    url=IAM_URL
)
wos_client = APIClient(
    authenticator=authenticator,
    service_url=SERVICE_URL,
    service_instance_id=None
)
print(wos_client.version)
#removed datamart

[Warning] No region provided : Using default region as us-south
3.1.4


<details>
<summary>▶ Explanation — What changed from the original</summary>

Creates the `wos_client` for interacting with Watson OpenScale (the monitoring service).

**Changes from the original:**
- `CloudPakForDataAuthenticator` import removed — not needed for IBM Cloud (only for Cloud Pak for Data on-premises)
- `service_instance_id = None` — tells the SDK to auto-detect the service instance
- `data_mart_id` line removed from here — the original used `wos_client.service_instance_id` which resolved to a stale ID. The data mart is now retrieved/created in a dedicated cell below.

</details>


### For computing answer quality and retrieval quality metrics you are required to configure LLM As Judge.

To compute metrics using LLM As Judge a generative_ai_evaluator integrated system should to be created and provided during prompt setup.

#### Create a Generative AI evaluator
The Generative AI Evaluator can be any model from watsonx.ai or a custom endpoint invoking external models

Supported evaluator types
* watsonx.ai (for connecting to watsonx.ai in local Cloud)
* custom (for connecting to any external models)

#### Integrated System parameters
| Parameter | Description |
|:-|:-|
| `name` | Name for the evaluator. |
| `description` | Description for the evaluator. |
| `type` | The type of integrated system. Provide `generative_ai_evaluator`. |
| `parameters` | The evaluator configuration details like `evaluator_type` and `model_id`. |
| `credentials` | The user credentials |
| `connection` [Optional]| The scoring endpoint details when the evaluator is of type `custom`. |

As an example, an evaluator using FLAN_UL2 model from watsonx.ai instance present in the same Cloud is created below. The other models which can be used from watsonx.ai are FLAN_T5_XXL, FLAN_UL2, FLAN_T5_XL, MIXTRAL_8X7B_INSTRUCT_V01.
For more details on the parameters and to create the other supported evaluators please refer to the link [Generative AI evaluator templates](https://github.ibm.com/aiopenscale/notebooks/wiki/Generative-AI-Evaluator-templates#for-ibm-watsonxgovernance-in-cloud)

In [41]:
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes

gen_ai_evaluator = wos_client.integrated_systems.add(
    name="RAG Metrics Evaluator",
    description="RAG Metrics Evaluator",
    type="generative_ai_evaluator",
    parameters={"evaluator_type": "watsonx.ai", "model_id": "meta-llama/llama-3-3-70b-instruct"},
    credentials={
        "wml_location": "cloud",
        "apikey": CLOUD_API_KEY,
    },
)

# get evaluator integrated system ID
result = gen_ai_evaluator.result._to_dict()
evaluator_id = result["metadata"]["id"]
evaluator_id

'019cf620-80f9-7213-a5ed-f7945886fe14'

<details>
<summary>▶ Explanation</summary>

Configures an LLM-as-a-judge evaluator. Some metrics (faithfulness, answer relevance, answer similarity, context relevance) require semantic understanding and cannot be computed with string matching — a second language model scores each response.

`meta-llama/llama-3-3-70b-instruct` is used as the judge. For each response, governance prompts this model with the question, context, and answer and asks it to score quality on a 1 to 10 scale.

`evaluator_id` is saved and passed to the monitor configuration in the prompt setup step.

</details>


#### Generative AI monitor parameters

##### Generative AI evaluator parameters
The generative_ai_evaluator details can be provided at the global level under `generative_ai_quality.parameters` to use the same evaluator for all the Answer quality metrics(Faithfulness, Answer relevance, Answer similarity) and Retrieval quality metrics(Context relevance, Retrieval precision, Average precision, Reciprocal rank, Hit rate, Normalized Discounted Cumulative Gain).

The generative ai evaluator can be specified at metric level to use different evaluators for each of the metric. The metrics under which generative_ai_evaluator parameter can be specified are faithfulness, answer_relevance, answer_similarity and retrieval_quality. The generative_ai_evaluator at the metric level takes precedence over the generative_ai_evaluator at global level

| Parameter | Description | Default Value |
|:-|:-|:-|
| `enabled`| The flag to enable generative ai evaluator |  |
| `evaluator_id`| The id of the generative ai evaluator integrated system. |  |

##### Faithfulness, Context relevance, Answer relevance and Answer similarity parameters

| Parameter | Description | Default Value |
|:-|:-|:-|
| `metric_prompt_template` [Optional]| The prompt template used to compute each metric values. User can override the prompt template used by watsonx.governance to compute the metric using this parameter. The prompt template should use the variables {context}, {question}, {answer}, {reference_answer} as needed and these variable values will be filled with the actual data while calling the scoring function. The prompt response should return the metric value in the range 1-10 for the respective metric and in one of the formats ["4", "7 star", "star: 8", "stars: 9"] as answer. |  |

##### Unsuccessful requests parameters

| Parameter | Description | Default Value |
|:-|:-|:-|
| `unsuccessful_phrases` [Optional]| The list of phrases to be used for comparing the model output to determine whether the request is unsuccessful or not. | `["i don't know", "i do not know", "i'm not sure", "i am not sure", "i'm unsure", "i am unsure", "i'm uncertain", "i am uncertain", "i'm not certain", "i am not certain", "i can't fulfill", "i cannot fulfill"]` |


## Configure Answer Quality metrics & Retrieval Quality metrics
<a id="config"></a>

### Parameters

#### Common parameters

| Parameter | Description | Default Value | Possible Value(s) |
|:-|:-|:-|:-|
| context_columns | The list of context column names in the input data frame. |  |  |
| question_column | the name of the question column in the input data frame. |  |  |
| answer_column | The name of the answer column in the input data frame |  |  |
| record_level | The flag to return the record level metrics values. Set the flag under configuration to generate record level metrics for all the metrics. Set the flag under specific metric to generate record level metrics for that metric alone. | `False` | `True`, `False` |

#### Faithfulness parameters
| Parameter | Description | Default Value | Possible Value(s) |
|:-|:-|:-|:-|
| attributions_count [Optional]| Source attributions are computed for each sentence in the generated answer. Source attribution for a sentence is the set of sentences in the context which contributed to the LLM generating that sentence in the answer.  The attributions_count parameter specifies the number of sentences in the context which need to be identified for attributions.  E.g., if the value is set to 2, then we will find the top 2 sentences from the context as source attributions. | `3` |  |
| ngrams [Optional]| The number of sentences to be grouped from the context when computing faithfulness score. These grouped sentences will be shown in the attributions. Having a very high value of ngrams might lead to having lower faithfulness scores due to dispersion of data and inclusion of unrelated sentences in the attributions. Having a very low value might lead to increase in metric computation time and attributions not capturing the all the aspects of the answer. | `2` |  |
| sample_size [Optional]| The faithfulness metric is computed for a maximum of 50 LLM responses.  If you wish to compute it for a smaller number of responses, set the sample_size value to a lower number. If the number of records in the input data frame are more than the sample size, a uniform random sample will taken for computation. | `50` | Integer between 0 to 50. Max value supported is 50 |
| record_level [Optional]| Set the flag to generate record level metrics for the specific metric. | `False` | `True`, `False` |


#### Context relevance parameters
| Parameter | Description | Default Value | Possible Value(s) |
|:-|:-|:-|:-|
| ngrams [Optional]| The number of sentences to be grouped from the context when computing context relevance score. Having a very high value of ngrams might lead to having lower context relevance scores due to dispersion of data and inclusion of unrelated sentences. | `5` |  |


#### Unsuccessful requests parameters
| Parameter | Description | Default Value |
|:-|:-|:-|
| unsuccessful_phrases [Optional]| The list of phrases to be used for comparing the model output to determine whether the request is unsuccessful or not. | `["i don't know", "i do not know", "i'm not sure", "i am not sure", "i'm unsure", "i am unsure", "i'm uncertain", "i am uncertain", "i'm not certain", "i am not certain", "i can't fulfill", "i cannot fulfill"]` |

### Configure faithfulness, answer relevance, and unsuccessful requests parameters

In [42]:
#CHECKING FOR DATAMART
data_marts = wos_client.data_marts.list()
print(data_marts.result.to_dict())

{'data_marts': [{'metadata': {'id': '042d1e9a-6c78-478c-a431-f2e0788984f1', 'crn': 'crn:v1:bluemix:public:aiopenscale:us-south:a/6152393966b5407b9bd9eebe01ee3fc4:042d1e9a-6c78-478c-a431-f2e0788984f1:data_mart:042d1e9a-6c78-478c-a431-f2e0788984f1', 'url': '/v2/data_marts/042d1e9a-6c78-478c-a431-f2e0788984f1', 'created_at': '2026-03-16T08:24:33.337000Z', 'created_by': 'IBMid-6A5000D5YE', 'modified_at': '2026-03-16T08:24:34.542000Z', 'modified_by': 'IBMid-6A5000D5YE', 'account_id': '6152393966b5407b9bd9eebe01ee3fc4'}, 'entity': {'name': 'RAG Data Mart', 'description': 'Data mart for Agentic RAG evaluation', 'service_instance_crn': 'crn:v1:bluemix:public:aiopenscale:us-south:a/6152393966b5407b9bd9eebe01ee3fc4:042d1e9a-6c78-478c-a431-f2e0788984f1::', 'internal_database': True, 'database_configuration': {'database_type': 'postgresql', 'name': 'internal database', 'credentials': {'secret_id': '019cf5bf-2f45-75eb-a31c-4cab9d63c316'}, 'location': {'schema_name': '042d1e9a-6c78-478c-a431-f2e0788

<details>
<summary>▶ Explanation</summary>

Lists existing data marts to check if one already exists before trying to create a new one. A data mart is the top-level container in watsonx.governance that holds all subscriptions and monitor instances.

</details>


In [43]:
# Try to get existing data mart first, create if none exists
existing = wos_client.data_marts.list().result.to_dict()

if existing["data_marts"]:
    data_mart_id = existing["data_marts"][0]["metadata"]["id"]
    print("Using existing data mart:", data_mart_id)
else:
    # Need to create one with internal_database=True
    response = wos_client.data_marts.add(
        background_mode=False,
        name="RAG Data Mart",
        description="Data mart for Agentic RAG evaluation",
        internal_database=True  # ← key fix, not database_configuration=None
    )
    result = response.result.to_dict()
    data_mart_id = result["metadata"]["id"]
    print("Created new data mart:", data_mart_id)

Using existing data mart: 042d1e9a-6c78-478c-a431-f2e0788984f1


<details>
<summary>▶ Explanation — Why this cell was added</summary>

The original notebook assumed a data mart already existed with a hardcoded ID. When that ID was not found, `execute_prompt_setup` failed with `AIQCS0100E: Resource of data_mart could not be found`.

This cell was added to handle both cases:
- If a data mart exists — uses the first one found
- If no data mart exists — creates one with `internal_database=True` which uses IBM's managed database rather than requiring you to provision your own PostgreSQL instance

</details>


In [44]:
label_column = "reference"
context_fields = ["context"]
question_field = "query"
operational_space_id = "development"
problem_type= "retrieval_augmented_generation"
input_data_type= "unstructured_text"

monitors = {
    "generative_ai_quality": {
        "parameters": {
            "generative_ai_evaluator": { # global LLM as judge configuration
               "enabled": True,
               "evaluator_id": evaluator_id,
            },
            "min_sample_size": 2,
            "metrics_configuration": {
                "faithfulness": {
                    # "metric_prompt_template": "", # adding custom template
                    # Uncomment generative_ai_evaluator to use a different evaluator for this metric.
                    # Takes higher precedence than the generative_ai_evaluator specified at global level.
                    # "generative_ai_evaluator": {  # metric specific LLM as judge configuration
                    #     "enabled": True,
                    #     "evaluator_id": evaluator_id,
                    # },
                },
                "answer_relevance": {
                    # "metric_prompt_template": "", # adding custom template
                    # Uncomment generative_ai_evaluator to use a different evaluator for this metric
                    # Takes higher precedence than the generative_ai_evaluator specified at global level.
                    # "generative_ai_evaluator": {  # metric specific LLM as judge configuration
                    #     "enabled": True,
                    #     "evaluator_id": evaluator_id,
                    # },
                },
                "rouge_score": {},
                "exact_match": {},
                "bleu": {},
                "unsuccessful_requests": {
                    # "unsuccessful_phrases": []
                },
                "hap_input_score": {},
                "hap_score": {},
                "pii": {},
                "pii_input": {},
                "retrieval_quality": {
                    # Uncomment generative_ai_evaluator to use a different evaluator for this metric
                    # Takes higher precedence than the generative_ai_evaluator specified at global level.
                    # "generative_ai_evaluator": {  # metric specific LLM as judge configuration
                    #     "enabled": True,
                    #     "evaluator_id": evaluator_id,
                    # },
                    # The metrics computed for retrieval quality are context_relevance, retrieval_precision, average_precision, reciprocal_rank, hit_rate, normalized_discounted_cumulative_gain
                    # "context_relevance": {
                    #     "metric_prompt_template": "", # adding custom template
                    # }
                },
                # Answer similarity metric is supported only when LLM as judge is configured. Uncomment only when using LLM as judge.
                "answer_similarity": {
                    # "metric_prompt_template": "", # adding custom template
                    # Uncomment generative_ai_evaluator to use a different evaluator for this metric
                    # Takes higher precedence than the generative_ai_evaluator specified at global level.
                    # "generative_ai_evaluator": {  # metric specific LLM as judge configuration
                    #     "enabled": True,
                    #     "evaluator_id": evaluator_id,
                    # },
                },
            },
        }
    }
}

response = wos_client.monitor_instances.mrm.execute_prompt_setup(
    prompt_template_asset_id=project_pta_id, 
    project_id=project_id,
    label_column=label_column,
    context_fields = context_fields,     
    question_field = question_field,     
    operational_space_id=operational_space_id, 
    problem_type=problem_type,
    input_data_type=input_data_type, 
    supporting_monitors=monitors, 
    background_mode=False
)

result = response.result
result.to_dict()

This method will be deprecated in the next release and be replaced by wos_client.wos.execute_prompt_setup() method



 Waiting for end of adding prompt setup c07c63ae-930f-4bd2-b2f6-9e541bc426de 




running.
finished

---------------------------------------------------------------
 Successfully finished setting up prompt template subscription 
---------------------------------------------------------------




{'prompt_template_asset_id': 'c07c63ae-930f-4bd2-b2f6-9e541bc426de',
 'project_id': '883c4e04-d048-4ac9-bf41-f6e25bbd6884',
 'deployment_id': '8df024a2-1dcc-5e14-8367-7a2c625db25f',
 'service_provider_id': '019cf5bf-7cc0-74ca-94a3-fa92ac86d794',
 'subscription_id': '019cf620-909c-7d96-b26c-c613d1966743',
 'mrm_monitor_instance_id': '019cf620-a3fd-7a88-bf1a-23285755b917',
 'start_time': '2026-03-16T10:10:53.629235Z',
 'end_time': '2026-03-16T10:11:07.906261Z',
 'status': {'state': 'FINISHED'}}

<details>
<summary>▶ Explanation — What each metric measures with simple examples</summary>

Sets up the monitoring subscription and configures which metrics to evaluate.

---

**Answer Quality Metrics** — how good is the answer?

| Metric | What it measures | Simple example |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context, or is the model hallucinating? | Context says "Leave is 20 days". Answer says "Leave is 25 days" → low faithfulness |
| **Answer Relevance** | Does the answer actually address the question asked? | Q: "What is the vacation policy?" A: "The CEO is John Smith" → low relevance |
| **Answer Similarity** | How semantically similar is the answer to the reference answer? | Requires LLM judge. Good if answer conveys the same meaning in different words |
| **BLEU** | Word n-gram overlap between answer and reference | Answer: "cats like fish" vs Reference: "cats love fish" → moderate BLEU |
| **ROUGE** | Similar to BLEU but recall-focused — how much of the reference appears in the answer | Standard summarisation metric |
| **Exact Match** | Does the answer exactly match the reference string? | Very strict — mostly useful for short factual answers |
| **Unsuccessful Requests** | What fraction of answers are "I don't know" type responses? | If model says "I'm not sure" → counted as unsuccessful |

---

**Retrieval Quality Metrics** — how good is the context retrieval?

| Metric | What it measures | Simple example |
|---|---|---|
| **Context Relevance** | Is the retrieved context actually relevant to the question? | Q: "What is the leave policy?" but context retrieved is about expenses → low relevance |
| **Retrieval Precision** | Of all retrieved chunks, what fraction are relevant? | Retrieved 5 chunks, 3 relevant → precision = 0.6 |
| **Average Precision** | Are the most relevant chunks ranked highest? | Relevant chunks at positions 1, 2, 5 scores higher than at 3, 4, 5 |
| **Reciprocal Rank** | Where is the first relevant chunk? | First relevant chunk at position 1 → score 1.0; at position 2 → score 0.5 |
| **Hit Rate** | Is there at least one relevant chunk in the results? | Binary: 1 if any relevant chunk retrieved, 0 if none |
| **NDCG** | Overall ranking quality of retrieved chunks | Penalises relevant chunks appearing lower in the ranked list |

---

**Content Safety Metrics**

| Metric | What it measures |
|---|---|
| **HAP Score** | Hate, Abuse, Profanity in the model **output** |
| **HAP Input Score** | Hate, Abuse, Profanity in the **input/question** |
| **PII** | Personally Identifiable Information in the **output** |
| **PII Input** | Personally Identifiable Information in the **input** |

</details>


With the following cell, you can read the prompt setup task and check its status

In [45]:
response = wos_client.monitor_instances.mrm.get_prompt_setup(
    prompt_template_asset_id=project_pta_id,
    project_id=project_id
)

result = response.result
result_json = result.to_dict()

if result_json["status"]["state"] == "FINISHED":
    print("Finished prompt setup. The response is {}".format(result_json))
else:
    print("Prompt setup failed. The response is {}".format(result_json))

This method will be deprecated in the next release and be replaced by wos_client.wos.get_prompt_setup() method
Finished prompt setup. The response is {'prompt_template_asset_id': 'c07c63ae-930f-4bd2-b2f6-9e541bc426de', 'project_id': '883c4e04-d048-4ac9-bf41-f6e25bbd6884', 'deployment_id': '8df024a2-1dcc-5e14-8367-7a2c625db25f', 'service_provider_id': '019cf5bf-7cc0-74ca-94a3-fa92ac86d794', 'subscription_id': '019cf620-909c-7d96-b26c-c613d1966743', 'mrm_monitor_instance_id': '019cf620-a3fd-7a88-bf1a-23285755b917', 'start_time': '2026-03-16T10:10:53.629235Z', 'end_time': '2026-03-16T10:11:07.906261Z', 'status': {'state': 'FINISHED'}}


<details>
<summary>▶ Explanation</summary>

Reads back the prompt setup result to confirm it finished successfully. Checks `result_json["status"]["state"] == "FINISHED"` — if it shows ERROR instead, the setup failed and the reason will be in `result_json["status"]["failure"]`.

</details>


In [46]:
print(result_json)

{'prompt_template_asset_id': 'c07c63ae-930f-4bd2-b2f6-9e541bc426de', 'project_id': '883c4e04-d048-4ac9-bf41-f6e25bbd6884', 'deployment_id': '8df024a2-1dcc-5e14-8367-7a2c625db25f', 'service_provider_id': '019cf5bf-7cc0-74ca-94a3-fa92ac86d794', 'subscription_id': '019cf620-909c-7d96-b26c-c613d1966743', 'mrm_monitor_instance_id': '019cf620-a3fd-7a88-bf1a-23285755b917', 'start_time': '2026-03-16T10:10:53.629235Z', 'end_time': '2026-03-16T10:11:07.906261Z', 'status': {'state': 'FINISHED'}}


In [47]:
subscription_id = result_json["subscription_id"]
mrm_monitor_instance_id = result_json["mrm_monitor_instance_id"]

<details>
<summary>▶ Explanation</summary>

Extracts the two key IDs needed for all subsequent governance operations:
- `subscription_id` — identifies this model's monitoring subscription
- `mrm_monitor_instance_id` — identifies the Model Risk Management monitor that runs risk evaluations

</details>


### Show all monitor instances in the development subscription
The following cell lists the monitors present in the development subscription, along with their respective statuses and other details. Please wait for all the monitors to be in an active state before proceeding further.

In [48]:
wos_client.monitor_instances.show(target_target_id=subscription_id)

042d1e9a-6c78-478c-a431-f2e0788984f1,active,019cf620-909c-7d96-b26c-c613d1966743,subscription,model_health,2026-03-16 10:10:59.829000+00:00,019cf620-a0bb-7baf-be1f-eeb2eda88d76
042d1e9a-6c78-478c-a431-f2e0788984f1,active,019cf620-909c-7d96-b26c-c613d1966743,subscription,generative_ai_quality,2026-03-16 10:10:58.894000+00:00,019cf620-9d30-7f64-b9ce-3d65af59dc1f
042d1e9a-6c78-478c-a431-f2e0788984f1,active,019cf620-909c-7d96-b26c-c613d1966743,subscription,mrm,2026-03-16 10:11:00.682000+00:00,019cf620-a3fd-7a88-bf1a-23285755b917


<details>
<summary>▶ Explanation</summary>

Lists all monitor instances attached to this subscription and their status. Wait until all monitors show `active` before running `evaluate_risk` — running evaluation before monitors are active can produce incomplete results.

</details>


### Risk evaluations for the PTA subscription - Evaluate the prompt template subscription

For risk assessment of a `development`-type subscription, you must have an evaluation dataset. The risk assessment function takes the evaluation dataset path as a parameter when evaluating the configured metrics. If there is a discrepancy between the feature columns in the subscription and the column names in the uploading `.CSV` file, you have the option to supply a mapping JSON file to associate the `.CSV` column names with the feature column names in the subscription.

**Note**: If you are running this notebook from Watson Studio, you may first need to upload your test data to Watson Studio, then run the code snippet below to download the feedback data file from the project to a local directory.

In [49]:
test_data_set_name = "data"
content_type = "multipart/form-data"
body = {}

# Preparing the test data, removing extra columns
cols_to_remove = ["uid", "doc", "title", "id"]
for col in cols_to_remove:
    if col in llm_data:
        del llm_data[col]
llm_data.to_csv(test_data_path, index=False)

response  = wos_client.monitor_instances.mrm.evaluate_risk(
    monitor_instance_id=mrm_monitor_instance_id,
    test_data_set_name=test_data_set_name, 
    test_data_path=test_data_path,
    content_type=content_type,
    body=body,
    project_id=project_id,
    includes_model_output=True,
    background_mode=False
)




 Waiting for risk evaluation of MRM monitor 019cf620-a3fd-7a88-bf1a-23285755b917 




upload_in_progress.
running....
finished

---------------------------------------
 Successfully finished evaluating risk 
---------------------------------------




<details>
<summary>▶ Explanation</summary>

Uploads `RAG_data.csv` to the monitor instance and triggers evaluation of all configured metrics. This can take 5 to 15 minutes.

- `background_mode=False` — waits for completion before the cell finishes
- `includes_model_output=True` — tells governance the CSV already contains model answers in `generated_text`, so it does not call the model again
- Extra columns (`uid`, `doc`, `title`, `id`) are removed first since they are not expected by the monitor schema

</details>


In [50]:
response  = wos_client.monitor_instances.mrm.get_risk_evaluation(mrm_monitor_instance_id, project_id=project_id)
response.result.to_dict()

{'metadata': {'id': '6c279d5a-6d2f-45a1-a7f8-f748f5a277df',
  'created_at': '2026-03-16T10:11:28.640Z',
  'created_by': 'iam-ServiceId-b317a8da-d926-496e-b0ca-6bcc57f556ae'},
 'entity': {'triggered_by': 'user',
  'parameters': {'evaluation_start_time': '2026-03-16T10:11:14.932527Z',
   'evaluator_user_key': '40ba627f-7c4a-47df-9d08-c9b8bb250890',
   'facts': {'state': 'finished'},
   'is_auto_evaluated': False,
   'measurement_id': '019cf621-135b-77ca-a77c-c8f7f4238fc1',
   'monitors_run_status': [{'monitor_id': 'generative_ai_quality',
     'status': {'state': 'finished'}},
    {'monitor_id': 'model_health', 'status': {'state': 'finished'}}],
   'project_id': '883c4e04-d048-4ac9-bf41-f6e25bbd6884',
   'prompt_template_asset_id': 'c07c63ae-930f-4bd2-b2f6-9e541bc426de',
   'user_iam_id': 'IBMid-6A5000D5YE',
   'wos_created_deployment_id': '8df024a2-1dcc-5e14-8367-7a2c625db25f',
   'publish_metrics': 'false',
   'evaluation_tests': ['drift_v2',
    'fairness',
    'generative_ai_quality'

<details>
<summary>▶ Explanation</summary>

Fetches the final risk evaluation result once processing completes. The returned dict shows which monitors ran, their status, and any errors. If state is ERROR, the failure details explain what went wrong.

</details>


### Display the Model Risk metrics.

Having calculated the measurements for the Foundation Model subscription, the Model Risk metrics generated for this subscription are available for your review:

In [51]:
wos_client.monitor_instances.show_metrics(monitor_instance_id=mrm_monitor_instance_id, project_id=project_id)

2026-03-16 10:11:28.731000+00:00,tests_passed,019cf621-135b-77ca-a77c-c8f7f4238fc1,1.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:28.731000+00:00,tests_run,019cf621-135b-77ca-a77c-c8f7f4238fc1,1.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:28.731000+00:00,tests_skipped,019cf621-135b-77ca-a77c-c8f7f4238fc1,4.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:28.731000+00:00,tests_failed,019cf621-135b-77ca-a77c-c8f7f4238fc1,0.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743


<details>
<summary>▶ Explanation</summary>

Displays the MRM (Model Risk Management) summary: `tests_run`, `tests_passed`, `tests_failed`, `tests_skipped`. These are high-level pass/fail counts based on thresholds — not the raw metric values. A test passes when all its configured metrics meet their lower/upper limit thresholds.

</details>


In [52]:
monitor_definition_id = "generative_ai_quality"
result = wos_client.monitor_instances.list(
    data_mart_id=data_mart_id,
    monitor_definition_id=monitor_definition_id,
    target_target_id=subscription_id,
    project_id=project_id
).result
result_json = result._to_dict()
genaiquality_monitor_id = result_json["monitor_instances"][0]["metadata"]["id"]
genaiquality_monitor_id

'019cf620-9d30-7f64-b9ce-3d65af59dc1f'

<details>
<summary>▶ Explanation</summary>

Looks up the `generative_ai_quality` monitor instance ID. The MRM and generative AI quality monitors are separate instances — this retrieves the quality monitor ID so its detailed metric scores can be displayed in the next cell.

</details>


## Display the Generative AI quality metrics

In [53]:
wos_client.monitor_instances.show_metrics(monitor_instance_id=genaiquality_monitor_id, project_id=project_id)

2026-03-16 10:11:58.340849+00:00,hap_input_score,019cf621-8704-741f-97ab-053fc77cadaf,0.0,None,0.0,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,rouge2,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.8,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,faithfulness,019cf621-8704-741f-97ab-053fc77cadaf,0.95,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,average_precision,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,records_processed,019cf621-8704-741f-97ab-053fc77cadaf,4.0,None,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,hit_rate,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,rougelsum,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.8,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,answer_relevance,019cf621-8704-741f-97ab-053fc77cadaf,0.75,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,hap_score,019cf621-8704-741f-97ab-053fc77cadaf,0.0,None,0.0,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,reciprocal_rank,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743


Note: First 10 records were displayed.


<details>
<summary>▶ Explanation — Final passing scores achieved</summary>

Displays the full set of computed metrics. After all fixes applied in this notebook, the final scores achieved were:

| Metric | Score | Threshold | Result |
|---|---|---|---|
| Faithfulness | 0.95 | 0.7 | Passed |
| Answer Relevance | 0.75 | 0.7 | Passed |
| Average Precision | 1.0 | 0.7 | Passed |
| Hit Rate | 1.0 | 0.7 | Passed |
| Reciprocal Rank | 1.0 | 0.7 | Passed |
| ROUGE-2 | 1.0 | 0.8 | Passed |
| ROUGE-Lsum | 1.0 | 0.8 | Passed |
| HAP Score | 0.0 | upper 0.0 | Passed |
| HAP Input Score | 0.0 | upper 0.0 | Passed |

You can also navigate to your watsonx.governance project in IBM Cloud and click the **Evaluate** tab on the prompt template asset to see these metrics as visual charts and dashboards.

</details>


## Congratulations!

You have completed this notebook. You can now navigate to the prompt template asset in your watsonx.governance project / space and click on the `Evaluate` tab to visualize the results in the UI.

In [54]:
response = wos_client.monitor_instances.mrm.get_risk_evaluation(
    mrm_monitor_instance_id, 
    project_id=project_id
)
print(json.dumps(response.result.to_dict(), indent=2))

{
  "metadata": {
    "id": "6c279d5a-6d2f-45a1-a7f8-f748f5a277df",
    "created_at": "2026-03-16T10:11:28.640Z",
    "created_by": "iam-ServiceId-b317a8da-d926-496e-b0ca-6bcc57f556ae"
  },
  "entity": {
    "triggered_by": "user",
    "parameters": {
      "evaluation_start_time": "2026-03-16T10:11:14.932527Z",
      "evaluator_user_key": "40ba627f-7c4a-47df-9d08-c9b8bb250890",
      "facts": {
        "state": "finished"
      },
      "is_auto_evaluated": false,
      "measurement_id": "019cf621-135b-77ca-a77c-c8f7f4238fc1",
      "monitors_run_status": [
        {
          "monitor_id": "generative_ai_quality",
          "status": {
            "state": "finished"
          }
        },
        {
          "monitor_id": "model_health",
          "status": {
            "state": "finished"
          }
        }
      ],
      "project_id": "883c4e04-d048-4ac9-bf41-f6e25bbd6884",
      "prompt_template_asset_id": "c07c63ae-930f-4bd2-b2f6-9e541bc426de",
      "user_iam_id": "IBMid-6A

In [55]:
wos_client.monitor_instances.show_metrics(
    monitor_instance_id=mrm_monitor_instance_id, 
    project_id=project_id
)

2026-03-16 10:11:28.731000+00:00,tests_passed,019cf621-135b-77ca-a77c-c8f7f4238fc1,1.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:28.731000+00:00,tests_run,019cf621-135b-77ca-a77c-c8f7f4238fc1,1.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:28.731000+00:00,tests_skipped,019cf621-135b-77ca-a77c-c8f7f4238fc1,4.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:28.731000+00:00,tests_failed,019cf621-135b-77ca-a77c-c8f7f4238fc1,0.0,None,None,['test_data_set_name:data'],mrm,019cf620-a3fd-7a88-bf1a-23285755b917,6c279d5a-6d2f-45a1-a7f8-f748f5a277df,subscription,019cf620-909c-7d96-b26c-c613d1966743


In [56]:
wos_client.monitor_instances.show_metrics(
    monitor_instance_id=genaiquality_monitor_id, 
    project_id=project_id
)

2026-03-16 10:11:58.340849+00:00,hap_input_score,019cf621-8704-741f-97ab-053fc77cadaf,0.0,None,0.0,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,rouge2,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.8,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,faithfulness,019cf621-8704-741f-97ab-053fc77cadaf,0.95,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,average_precision,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,records_processed,019cf621-8704-741f-97ab-053fc77cadaf,4.0,None,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,hit_rate,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,rougelsum,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.8,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,answer_relevance,019cf621-8704-741f-97ab-053fc77cadaf,0.75,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,hap_score,019cf621-8704-741f-97ab-053fc77cadaf,0.0,None,0.0,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743
2026-03-16 10:11:58.340849+00:00,reciprocal_rank,019cf621-8704-741f-97ab-053fc77cadaf,1.0,0.7,None,"['computed_on:feedback', 'field_type:subscription', 'aggregation_type:mean']",generative_ai_quality,019cf620-9d30-7f64-b9ce-3d65af59dc1f,9a7ad3d4-3d75-4863-abf6-c8cc30cd4b94,subscription,019cf620-909c-7d96-b26c-c613d1966743


Note: First 10 records were displayed.


In [60]:
for i, row in llm_data.iterrows():
    print(f"Q: {row['query']}")
    print(f"A: {row['generated_text'][:300]}")
    print()

Q: What is compliance with law and firm policies for HR process?
A: According to the JP Morgan Chase Code of Conduct documents, compliance with law and firm policies in HR processes involves adhering to all applicable laws, regulations, and internal policies. This includes respecting equal employment opportunity laws, anti-discrimination and anti-harassment policies

Q: What are considered as Prohibited AI systems?
A: According to the high-level summary of the EU AI Act, the following types of AI systems are considered prohibited:

1. AI systems deploying subliminal, manipulative, or deceptive techniques to distort behavior and impair informed decision-making, causing significant harm.
2. AI systems exploiting vu

Q: What are the obligations for high-risk AI systems under the EU AI Act?
A: According to the high-level summary of the EU AI Act, the following types of AI systems are considered prohibited:

1. AI systems deploying subliminal, manipulative, or deceptive techniques to distor

In [58]:
import inspect
print(inspect.getsource(run_agent))


def run_agent(input_text: str, session_id: str = "default"):
    config = {"configurable": {"thread_id": session_id}}
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": input_text}]},
        config=config
    )
    output = ""
    for msg in reversed(result["messages"]):
        if hasattr(msg, "type") and msg.type == "ai":
            if isinstance(msg.content, str) and msg.content.strip():
                if not msg.content.strip().startswith('[{"name"') and "tool_response" not in msg.content:
                    output = msg.content
                    break
            elif isinstance(msg.content, list):
                text_parts = [p["text"] for p in msg.content 
                              if isinstance(p, dict) and p.get("type") == "text" and p.get("text","").strip()
                              and not p["text"].strip().startswith('[{"name"')]
                if text_parts:
                    output = " ".join(text_parts)
               

In [59]:
print("All Tests run")

All Tests run
